In [10]:
# SECTION 1: Verify environment
#===============================================
import sys
import torch
import transformers
import datasets
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns

print("=== Environment Check ===")
print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers   : {transformers.__version__}")
print(f"Datasets       : {datasets.__version__}")
print(f"Pandas         : {pd.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")  # Will print False on CPU
print("All imports successful")

=== Environment Check ===
Python version : 3.12.10
PyTorch version: 2.12.0+cpu
Transformers   : 4.40.1
Datasets       : 2.19.0
Pandas         : 3.0.3
CUDA available : False
All imports successful


In [4]:
# SECTION 2: Load Dataset (Bitext)
#===============================================
from datasets import load_dataset

print("Loading Bitext dataset from HuggingFace Hub...")

raw_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train"
)

print(f"\nDataset loaded successfully ✓")
print(f"Total rows    : {len(raw_dataset):,}")
print(f"Column names  : {raw_dataset.column_names}")
print(f"\nSample row:")
print(raw_dataset[0])

Loading Bitext dataset from HuggingFace Hub...



Dataset loaded successfully ✓
Total rows    : 26,872
Column names  : ['flags', 'instruction', 'category', 'intent', 'response']

Sample row:
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [5]:
# SECTION 3: Explore the Dataset
#===============================================
df_full = raw_dataset.to_pandas()
print("=== Dataset Overview ===")
print(f"Shape: {df_full.shape}")
print(f"\nColumn dtypes:\n{df_full.dtypes}")
print(f"\nNull values:\n{df_full.isnull().sum()}")

# Show unique intents and categories?
print(f"\nUnique intents   : {df_full['intent'].nunique()}")
print(f"Unique categories: {df_full['category'].nunique()}")

# Show distribution of categories
print(f"\nSamples per category:")
print(df_full['category'].value_counts().to_string())

=== Dataset Overview ===
Shape: (26872, 5)

Column dtypes:
flags          str
instruction    str
category       str
intent         str
response       str
dtype: object

Null values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Unique intents   : 27
Unique categories: 11

Samples per category:
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950


In [6]:
# SECTION 4: Create an evaluation sample
#===============================================
SAMPLE_PER_CATEGORY = 10
RANDOM_SEED = 42 

def create_stratified_sample(
    dataframe: pd.DataFrame,
    group_column: str,
    n_per_group: int,
    random_seed: int
) -> pd.DataFrame:
    sampled_groups = []

    for group_value, group_df in dataframe.groupby(group_column):
        n_to_sample = min(n_per_group, len(group_df))
        sample = group_df.sample(n=n_to_sample, random_state=random_seed)
        sampled_groups.append(sample)

    return pd.concat(sampled_groups, ignore_index=True)

df_sample = create_stratified_sample(
    dataframe=df_full,
    group_column="category",
    n_per_group=SAMPLE_PER_CATEGORY,
    random_seed=RANDOM_SEED
)

print(f"Stratified sample created: {len(df_sample)} rows")
print(f"Categories covered       : {df_sample['category'].nunique()}")
print(f"\nSamples per category:\n{df_sample['category'].value_counts().to_string()}")

Stratified sample created: 110 rows
Categories covered       : 11

Samples per category:
category
ACCOUNT         10
CANCEL          10
CONTACT         10
DELIVERY        10
FEEDBACK        10
INVOICE         10
ORDER           10
PAYMENT         10
REFUND          10
SHIPPING        10
SUBSCRIPTION    10


In [8]:
# SECTION 5: Save sample to disk
#===============================================
import os

os.makedirs("../data", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../outputs/plots", exist_ok=True)

sample_path = "../data/sample_bitext.csv"
df_sample.to_csv(sample_path, index=False)

print(f"Sample saved to: {sample_path}")
print(f"This file will be loaded by all subsequent notebooks.")
print(f"\nFirst 3 rows preview:")
df_sample[["instruction", "response", "intent", "category"]].head(3)

Sample saved to: ../data/sample_bitext.csv
This file will be loaded by all subsequent notebooks.

First 3 rows preview:


,instruction,response,intent,category
0,where to notify of problems with sign-up?,Glad you contacted to us regarding the problem...,registration_problems,ACCOUNT
1,wanna use the {{Account Type}} profile help me,I'll get right on it! I'm thrilled to assist y...,switch_account,ACCOUNT
2,I cannot retrieve the bloody PIN code of my pr...,I'm sorry to hear that you're having trouble r...,recover_password,ACCOUNT


In [9]:
print(df_sample[["instruction", "response", "intent", "category"]].head(3).to_string())

                                           instruction                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           response                 intent categor